#### ============================================
###  1. Import Libraries
#### ============================================

In [1]:
import pandas as pd
import numpy as np
import joblib
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

#### ============================================
###  2. Read a Data
#### ============================================

In [2]:
Data = pd.read_csv('/Users/mac/Desktop/Zain Projects/Task 2 Visitor Mall Dataset/data/processed/total_visitors_per_mall_cleaned.csv')

#### ============================================
###  3. Check a Data
#### ============================================

In [3]:
Data.head()

,SUBSCRIBER_ID,VISIT_DATE,MALL_NAME,HOME_GOVERNORATE,WORK_GOVERNORATE,NATIONALITY,ARPU_BRACKET,DEVICE_OS,AGE_BRACKET,GENDER,STUDENT,IS_EMPLOYEE,CREDIT_CATEGORY,YEAR,DAY
0,11121,2022-11-08 13:18:15,Sunset Mall,Governorate 1,Governorate 1,Citizen,Low,iOS,> 25 and <= 45,Female,Not a student,0,Fair,2022,8
1,11122,2022-11-19 15:28:46,City Center Mall,Governorate 2,Governorate 2,Citizen,High,iOS,> 25 and <= 45,Male,Not a student,0,Excellent,2022,19
2,11122,2022-11-04 11:44:39,Sunset Mall,Governorate 2,Governorate 2,Citizen,High,iOS,> 25 and <= 45,Male,Not a student,0,Excellent,2022,4
3,11123,2022-11-05 14:52:57,City Center Mall,Governorate 1,Governorate 1,Citizen,Low,iOS,<= 18,Male,Not a student,0,Excellent,2022,5
4,11124,2022-11-07 17:43:37,Oasis Mall,Governorate 2,Governorate 2,Citizen,High,iOS,> 45,Male,Not a student,0,Excellent,2022,7


In [4]:
Data['VISIT_DATE'] = pd.to_datetime(Data['VISIT_DATE'])
Data['HOUR_TS'] = Data['VISIT_DATE'].dt.floor('h')

In [8]:
print(f" Number of Rows:{len(Data)}")

 Number of Rows:899536


In [10]:
print(f"Malls:{Data["MALL_NAME"].unique().tolist()}")

Malls:['Sunset Mall', 'City Center Mall', 'Oasis Mall', 'Century mall', 'Rainbow mall']


In [12]:
print(f"Time Period: {Data['VISIT_DATE'].min()} → {Data['VISIT_DATE'].max()}")

Time Period: 2022-11-01 00:00:00 → 2022-11-30 23:59:50


#### ============================================
###  4. Hourly Data Aggregation per Mall
#### ============================================

In [14]:
hourly = Data.groupby(['MALL_NAME', 'HOUR_TS']).size().reset_index(name='visits')

In [15]:
# Create a full grid for each Mall × each Hour (to fill missing hours with zero)
malls = hourly['MALL_NAME'].unique()
full_range = pd.date_range(hourly['HOUR_TS'].min(), hourly['HOUR_TS'].max(), freq='h')
grid = pd.MultiIndex.from_product(
    [malls, full_range], names=['MALL_NAME', 'HOUR_TS']
).to_frame(index=False)

hourly = grid.merge(hourly, on=['MALL_NAME', 'HOUR_TS'], how='left').fillna({'visits': 0})

In [16]:
print(f"Post-aggregation: {len(hourly):,} rows ({len(malls)} malls × {len(full_range)} hours)")

Post-aggregation: 3,537 rows (5 malls × 720 hours)


#### ============================================
###  5. Feature Engineering
#### ============================================

In [17]:
hourly['hour']       = hourly['HOUR_TS'].dt.hour
hourly['dayofweek']  = hourly['HOUR_TS'].dt.dayofweek   # 0=Monday
hourly['day']        = hourly['HOUR_TS'].dt.day
hourly['is_weekend'] = hourly['dayofweek'].isin([4, 5]).astype(int)

In [18]:
hourly['hour_sin'] = np.sin(2 * np.pi * hourly['hour'] / 24)
hourly['hour_cos'] = np.cos(2 * np.pi * hourly['hour'] / 24)
hourly['dow_sin']  = np.sin(2 * np.pi * hourly['dayofweek'] / 7)
hourly['dow_cos']  = np.cos(2 * np.pi * hourly['dayofweek'] / 7)

In [19]:
hourly = hourly.sort_values(['MALL_NAME', 'HOUR_TS'])
hourly['lag_1']   = hourly.groupby('MALL_NAME')['visits'].shift(1)
hourly['lag_24']  = hourly.groupby('MALL_NAME')['visits'].shift(24)
hourly['lag_168'] = hourly.groupby('MALL_NAME')['visits'].shift(168)

In [20]:
hourly['rolling_24h'] = hourly.groupby('MALL_NAME')['visits'].transform(
    lambda x: x.shift(1).rolling(24, min_periods=1).mean()
)

In [21]:
hourly = pd.get_dummies(hourly, columns=['MALL_NAME'], prefix='mall')

In [23]:
hourly = hourly.dropna().reset_index(drop=True)
 
FEATURE_COLS = [c for c in hourly.columns if c not in ['HOUR_TS', 'visits']]
print(f" Features: {len(FEATURE_COLS)}")
print(f"Features: {FEATURE_COLS}")

 Features: 17
Features: ['hour', 'dayofweek', 'day', 'is_weekend', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'lag_1', 'lag_24', 'lag_168', 'rolling_24h', 'mall_Century mall', 'mall_City Center Mall', 'mall_Oasis Mall', 'mall_Rainbow mall', 'mall_Sunset Mall']


In [24]:
Data

,SUBSCRIBER_ID,VISIT_DATE,MALL_NAME,HOME_GOVERNORATE,WORK_GOVERNORATE,NATIONALITY,ARPU_BRACKET,DEVICE_OS,AGE_BRACKET,GENDER,STUDENT,IS_EMPLOYEE,CREDIT_CATEGORY,YEAR,DAY,HOUR_TS
0,11121,2022-11-08 13:18:15,Sunset Mall,Governorate 1,Governorate 1,Citizen,Low,iOS,> 25 and <= 45,Female,Not a student,0,Fair,2022,8,2022-11-08 13:00:00
1,11122,2022-11-19 15:28:46,City Center Mall,Governorate 2,Governorate 2,Citizen,High,iOS,> 25 and <= 45,Male,Not a student,0,Excellent,2022,19,2022-11-19 15:00:00
2,11122,2022-11-04 11:44:39,Sunset Mall,Governorate 2,Governorate 2,Citizen,High,iOS,> 25 and <= 45,Male,Not a student,0,Excellent,2022,4,2022-11-04 11:00:00
3,11123,2022-11-05 14:52:57,City Center Mall,Governorate 1,Governorate 1,Citizen,Low,iOS,<= 18,Male,Not a student,0,Excellent,2022,5,2022-11-05 14:00:00
4,11124,2022-11-07 17:43:37,Oasis Mall,Governorate 2,Governorate 2,Citizen,High,iOS,> 45,Male,Not a student,0,Excellent,2022,7,2022-11-07 17:00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
899531,378366,2022-11-15 19:14:14,Century mall,Governorate 1,Governorate 1,Citizen,High,Android,> 45,Male,Not a student,0,Excellent,2022,15,2022-11-15 19:00:00
899532,378366,2022-11-27 15:05:58,Century mall,Governorate 1,Governorate 1,Citizen,High,Android,> 45,Male,Not a student,0,Excellent,2022,27,2022-11-27 15:00:00
899533,378366,2022-11-24 14:14:29,City Center Mall,Governorate 1,Governorate 1,Citizen,High,Android,> 45,Male,Not a student,0,Excellent,2022,24,2022-11-24 14:00:00
899534,378366,2022-11-24 21:24:54,City Center Mall,Governorate 1,Governorate 1,Citizen,High,Android,> 45,Male,Not a student,0,Excellent,2022,24,2022-11-24 21:00:00


#### ============================================
###  6. Split a Data
#### ============================================

In [25]:
hourly = hourly.sort_values('HOUR_TS').reset_index(drop=True)

In [26]:
n_malls = len([c for c in hourly.columns if c.startswith('mall_')])
split_idx = len(hourly) - 168 * n_malls

In [27]:
X = hourly[FEATURE_COLS]
y = hourly['visits']

In [28]:
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

In [29]:
print(f"Train size: {len(X_train):,} صف")
print(f"Test size : {len(X_test):,} صف (آخر 7 أيام)")

Train size: 1,857 صف
Test size : 840 صف (آخر 7 أيام)


#### ============================================
###  7. Build a Model
#### ============================================

In [31]:
model = RandomForestRegressor(
    n_estimators=300,
    max_depth=12,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1     
)

###  7.1 Training a Model

In [32]:
model.fit(X_train, y_train)

RandomForestRegressor(max_depth=12, min_samples_leaf=2, n_estimators=300,
                      n_jobs=-1, random_state=42)

###  7.2 Evaluation a Model

In [34]:
preds = np.clip(model.predict(X_test), 0, None)
 
mae  = mean_absolute_error(y_test, preds)
rmse = np.sqrt(mean_squared_error(y_test, preds))
r2   = r2_score(y_test, preds)

In [39]:
print(f"MAE:  {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R²:   {r2:.4f}")

MAE:  29.22
RMSE: 65.33
R²:   0.9485


{'RMSE: 65.33'}


In [36]:
print(f"MAE  (متوسط الخطأ المطلق) : {mae:.2f} زائر")
print(f"RMSE (جذر متوسط مربع الخطأ): {rmse:.2f} زائر")
print(f"R²   (دقة الموديل)         : {r2:.3f}  (1.0 = مثالي)")

MAE  (متوسط الخطأ المطلق) : 29.22 زائر
RMSE (جذر متوسط مربع الخطأ): 65.33 زائر
R²   (دقة الموديل)         : 0.949  (1.0 = مثالي)


#### ============================================
###  8. Save a Model
#### ============================================

In [40]:
joblib.dump(model,        'footfall_model.pkl')
joblib.dump(FEATURE_COLS, 'feature_cols.pkl')

['feature_cols.pkl']

In [41]:
hourly.to_csv('hourly_features.csv', index=False)